# YOLO26n-Depth: Joint Training Pipeline
Pipeline training ini memungkinkan Anda untuk melatih model secara end-to-end untuk tugas *object detection* dan *monocular depth estimation* secara bersamaan.

Pendekatan *multi-task* dengan *shared backbone* ini sangat efisien dalam menekan *computational load*. Hal ini sangat ideal jika arsitektur ini nantinya di-*deploy* ke *Single Board Computer* atau *edge AI platforms* (seperti NVIDIA Jetson) karena tidak perlu mengeksekusi dua model secara terpisah.

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import numpy as np
import os

# Import arsitektur yang sudah diperbarui
# Pastikan file arsitektur.py berada dalam satu direktori dengan notebook ini
from arsitektur import build_model

## 1. Dataloader Kustom (Mendukung SUNRGBD & COCO)
Strategi untuk melatih dengan dua dataset berbeda: SUNRGBD memiliki label *bounding box* dan peta kedalaman (*depth map*), sementara COCO hanya memiliki *bounding box*.

Kelas `Dataset` ini dirancang dinamis. Jika dataset tidak memiliki *depth map* (seperti COCO), sistem akan mengembalikan *flag* `has_depth = False`. *Flag* ini nantinya berguna untuk memberi sinyal pada fungsi *loss* agar tidak melakukan propagasi mundur (*backward pass*) pada bagian *depth head* saat gambar tersebut diproses.

In [10]:
class JointVisionDataset(Dataset):
    def __init__(self, img_dir, label_dir, depth_dir=None, img_size=(640, 640)):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.depth_dir = depth_dir  # Kosongkan atau set None jika melatih COCO
        self.img_size = img_size
        self.img_files = sorted(os.listdir(img_dir)) if os.path.exists(img_dir) else []
        
    def __len__(self):
        return len(self.img_files)
    
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        
        # 1. Load & Preprocess RGB Image
        img_path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None: img = np.zeros((*self.img_size, 3), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, self.img_size)
        img = img.astype(np.float32) / 255.0  # Normalisasi ke [0, 1]
        img = np.transpose(img, (2, 0, 1))    # Konversi HWC ke CHW untuk PyTorch
        
        # 2. Load Depth Map (khusus SUNRGBD)
        has_depth = False
        if self.depth_dir is not None:
            # Format nama disesuaikan dengan struktur dataset Anda
            depth_path = os.path.join(self.depth_dir, img_name.replace('.jpg', '.png'))
            if os.path.exists(depth_path):
                depth = cv2.imread(depth_path, cv2.IMREAD_ANYDEPTH)
                depth = cv2.resize(depth, self.img_size)
                depth = depth.astype(np.float32)
                # Normalisasi depth relatif [0, 1]
                depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-6)
                has_depth = True
                
        if not has_depth:
            # Fallback tensor kosong untuk COCO agar batching tidak error
            depth = np.zeros(self.img_size, dtype=np.float32)
            
        depth = np.expand_dims(depth, axis=0) # 1HW format
        
        # 3. Load Labels (YOLO format: class, cx, cy, w, h)
        # [! PLACEHOLDER !] Bagian ini perlu parser spesifik (baca txt/json) 
        # bergantung pada format bounding box yang Anda miliki.
        bboxes = torch.zeros((0, 5))
        
        return torch.tensor(img), torch.tensor(bboxes), torch.tensor(depth), torch.tensor(has_depth)


## 2. Multi-Task Loss Function
Total *loss* yang dievaluasi adalah kombinasi dinamis antara *Detection Loss* (CIoU + Classification) dan *Depth Loss* (biasanya *Smooth L1 Loss*). Pembobotan (*weighting*) bisa dieksplorasi secara eksperimental.

In [11]:
class YOLO26MultiTaskLoss(nn.Module):
    def __init__(self, w_det=1.0, w_depth=0.5):
        super().__init__()
        self.w_det = w_det
        self.w_depth = w_depth
        self.depth_criterion = nn.SmoothL1Loss()
        
    def forward(self, preds, targets_bbox, targets_depth, has_depth):
        # --- 1. Depth Loss ---
        # Hanya hitung error kedalaman jika gambar memiliki ground-truth depth map
        pred_depth = preds['depth']
        if has_depth.any():
            valid_pred_depth = pred_depth[has_depth]
            valid_target_depth = targets_depth[has_depth]
            loss_depth = self.depth_criterion(valid_pred_depth, valid_target_depth)
        else:
            # Return 0 gradient jika batch saat ini (misal COCO) murni deteksi 2D
            loss_depth = torch.tensor(0.0, device=pred_depth.device, requires_grad=True)
            
        # --- 2. Detection Loss ---
        # Placeholder: Di sinilah letak implementasi CIoU / BBox Regression 
        # dan target matching untuk P3, P4, P5.
        loss_det = torch.tensor(0.0, device=pred_depth.device, requires_grad=True)
        
        total_loss = (self.w_det * loss_det) + (self.w_depth * loss_depth)
        return total_loss, loss_det, loss_depth


## 3. Training Loop Utama

In [ ]:
def train_yolo26(model, dataloader, epochs=30, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Memulai training di device: {device}")
    
    model = model.to(device)
    model.train()
    
    criterion = YOLO26MultiTaskLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        for batch_idx, (imgs, bboxes, depths, has_depth) in enumerate(dataloader):
            imgs = imgs.to(device)
            depths = depths.to(device)
            has_depth = has_depth.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass (Penting: Set NMS=False selama proses training)
            preds = model(imgs, use_nms=False)
            
            # Kalkulasi Multi-Task Loss
            loss, l_det, l_depth = criterion(preds, bboxes, depths, has_depth)
            
            # Backward pass & update bobot
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
            if batch_idx % 10 == 0:
                print(f"Epoch {epoch+1}/{epochs} | Batch {batch_idx}/{len(dataloader)} | "
                      f"Total Loss: {loss.item():.4f} (Det: {l_det.item():.4f}, Depth: {l_depth.item():.4f})")
                
        print(f"[!] Selesai Epoch {epoch+1} | Rata-rata Loss: {epoch_loss/len(dataloader):.4f}\n")

# =========================================
# BLOK EKSEKUSI (Uncomment untuk menjalankan)
# =========================================

# 1. Inisialisasi Model (Contoh 80 kelas jika mencakup COCO)
model = build_model(num_classes=80)

# 2. Persiapkan Dataloader (Ganti dengan path direktori laptop Anda)
dataset = JointVisionDataset(img_dir='data/images', label_dir='data/labels', depth_dir='data/depths')
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=4)

# 3. Eksekusi Training
train_yolo26(model, dataloader, epochs=30, lr=1e-3)
